# Phase 2 — Exp10 + Exp11 + Exp12 (All non-token pairwise fusions)

This notebook runs the experiments together using one common preprocessing/training pipeline.

In [3]:

# Phase 2 — Common setup for paired experiments
# Exp8–9 or Exp10–12
#
# Expected files in DATA_DIR:
#   train.json / train.jsonl  (or adjust TRAIN_PATH below)
#   test.json / test.jsonl    (not used for validation)
#
# Each record must contain:
#   text  : token sequence (list of tokens, or a whitespace-separated string)
#   label : "A" / "B"
#
# Canonical validation protocol:
#   80/20 stratified split, random_state=42
#
# IMPORTANT:
#   This notebook only compares experiments on the validation split.
#   It does NOT retrain on all 10,536 examples and does NOT create a Kaggle
#   submission. Do that only after selecting a final experiment.

import os, re, json, time, random, zipfile, warnings
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ---- Adjust this path if necessary ----
DATA_DIR = "/content/data"
TRAIN_PATH = None  # e.g. "/content/data/train.json"
OUTPUT_DIR = "/content/phase2_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_LEN = 384
EMB_DIM = 128
CNN_CHANNELS = 128
Z_DIM = 128
BATCH_SIZE = 64
EPOCHS = 15
PATIENCE = 3
LR = 2e-3
WEIGHT_DECAY = 1e-4

TFIDF_MAX_FEATURES = 200_000
TFIDF_MIN_DF = 3
TRANS_MAX_FEATURES = 50_000
TRANS_MIN_DF = 2
SVD_DIM = 256

def _find_file(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

if TRAIN_PATH is None:
    TRAIN_PATH = _find_file([
        os.path.join(DATA_DIR, "train.json"),
        os.path.join(DATA_DIR, "train.jsonl"),
        "/content/train.json",
        "/content/train.jsonl",
        "./train.json",
        "./train.jsonl",
    ])

if TRAIN_PATH is None:
    raise FileNotFoundError(
        "Set TRAIN_PATH to your labelled training file. "
        "Expected JSON/JSONL records with text and label."
    )

def load_records(path):
    # If the file is .jsonl or specifically train.json (known to be JSON Lines),
    # load it line by line.
    if path.endswith(".jsonl") or os.path.basename(path) == "train.json":
        with open(path, "r", encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]
    # Otherwise, try to load it as a single JSON object.
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for k in ("data", "records", "train"):
            if k in obj and isinstance(obj[k], list):
                return obj[k]
    raise ValueError("Unsupported JSON structure.")

records = load_records(TRAIN_PATH)
df = pd.DataFrame(records)

def normalize_tokens(x):
    if isinstance(x, list):
        return [str(t) for t in x]
    if isinstance(x, str):
        return x.split()
    return [str(t) for t in x]

df["tokens"] = df["text"].apply(normalize_tokens)
df["label_num"] = df["label"].map({"A": 1, "B": 0}).astype(int)

train_idx, val_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    stratify=df["label_num"].values,
    random_state=SEED,
)

train_idx = np.asarray(train_idx)
val_idx = np.asarray(val_idx)

print("Total:", len(df))
print("Train:", len(train_idx), "Validation:", len(val_idx))
print("Train label counts:", Counter(df.iloc[train_idx]["label"].values))
print("Val label counts:", Counter(df.iloc[val_idx]["label"].values))


Device: cuda
Total: 10536
Train: 8428 Validation: 2108
Train label counts: Counter({'B': 5469, 'A': 2959})
Val label counts: Counter({'B': 1368, 'A': 740})


In [4]:

def entropy_from_counts(counts):
    c = np.asarray(counts, dtype=float)
    c = c[c > 0]
    if len(c) == 0:
        return 0.0
    p = c / c.sum()
    return float(-(p * np.log2(p)).sum())

def sequence_features(doc):
    x = list(doc)
    n = len(x)
    if n == 0:
        return np.zeros(62, dtype=np.float32)

    cnt = Counter(x)
    uniq = len(cnt)

    bigrams = list(zip(x[:-1], x[1:]))
    trigrams = list(zip(x[:-2], x[1:-1], x[2:]))

    def div(seq):
        return len(set(seq)) / max(1, len(seq))

    def rep(seq):
        return 1.0 - div(seq)

    half = max(1, n // 2)
    q = max(1, n // 4)

    first = x[:half]
    second = x[half:]
    q1, q2, q3, q4 = x[:q], x[q:2*q], x[2*q:3*q], x[3*q:]

    feats = [
        n,
        uniq,
        uniq / n,
        len([v for v in cnt.values() if v > 1]) / n,
        max(cnt.values()) / n,
        np.mean(list(cnt.values())),
        np.std(list(cnt.values())),
        entropy_from_counts(list(cnt.values())),
        len(bigrams),
        len(set(bigrams)),
        div(bigrams),
        rep(bigrams),
        len(trigrams),
        len(set(trigrams)),
        div(trigrams),
        rep(trigrams),
        sum(x[i] == x[i-1] for i in range(1, n)) / max(1, n-1),
        len(set(zip(x[:-1], x[1:]))) / max(1, n-1),
        entropy_from_counts(list(Counter(bigrams).values())),
        entropy_from_counts(list(Counter(trigrams).values())),
        len(set(first)) / len(first),
        len(set(second)) / max(1, len(second)),
        entropy_from_counts(list(Counter(first).values())),
        entropy_from_counts(list(Counter(second).values())),
        len(set(q1)) / len(q1),
        len(set(q2)) / len(q2),
        len(set(q3)) / len(q3),
        len(set(q4)) / max(1, len(q4)),
        entropy_from_counts(list(Counter(q1).values())),
        entropy_from_counts(list(Counter(q2).values())),
        entropy_from_counts(list(Counter(q3).values())),
        entropy_from_counts(list(Counter(q4).values())),
        len(first),
        len(second),
        len(q1),
        len(q2),
        len(q3),
        len(q4),
        len(set(first) & set(second)) / max(1, len(set(first) | set(second))),
        len(set(q1) & set(q2)) / max(1, len(set(q1) | set(q2))),
        len(set(q2) & set(q3)) / max(1, len(set(q2) | set(q3))),
        len(set(q3) & set(q4)) / max(1, len(set(q3) | set(q4))),
        np.mean([len(t) for t in x]),
        np.std([len(t) for t in x]),
        min([len(t) for t in x]),
        max([len(t) for t in x]),
        np.mean([len(t) for t in x[:half]]),
        np.mean([len(t) for t in x[half:]]) if second else 0.0,
        len(x[0]),
        len(x[-1]),
        len(set(x[:min(10,n)])) / min(10,n),
        len(set(x[max(0,n-10):])) / min(10,n),
        entropy_from_counts(list(Counter(x[:min(10,n)]).values())),
        entropy_from_counts(list(Counter(x[max(0,n-10):]).values())),
        len(set(x[:min(25,n)])) / min(25,n),
        len(set(x[max(0,n-25):])) / min(25,n),
        entropy_from_counts(list(Counter(x[:min(25,n)]).values())),
        entropy_from_counts(list(Counter(x[max(0,n-25):]).values())),
    ]
    assert len(feats) == 62, len(feats)
    return np.asarray(feats, dtype=np.float32)


In [8]:

class TokenDataset(Dataset):
    def __init__(self, sequences, labels):
        self.x = sequences
        self.y = np.asarray(labels, dtype=np.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return torch.tensor(self.x[i], dtype=torch.long), torch.tensor(self.y[i])

def build_vocab(all_docs):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for doc in all_docs:
        for t in doc:
            if t not in vocab:
                vocab[t] = len(vocab)
    return vocab

def encode_docs(docs, vocab):
    out = []
    for doc in docs:
        ids = [vocab.get(t, 1) for t in doc[:MAX_LEN]]
        ids += [0] * (MAX_LEN - len(ids))
        out.append(ids)
    return np.asarray(out, dtype=np.int64)

def make_transition_strings(docs):
    all_transitions = []
    for doc in docs:
        transitions = []
        for i in range(len(doc) - 1):
            transitions.append(f"{doc[i]}_{doc[i+1]}")
        all_transitions.append(" ".join(transitions))
    return all_transitions


class TokenEncoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, EMB_DIM, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(EMB_DIM, CNN_CHANNELS, k, padding=k//2)
            for k in (3, 5, 7)
        ])
        self.proj = nn.Sequential(
            nn.Linear(CNN_CHANNELS * 3, Z_DIM),
            nn.GELU(),
            nn.LayerNorm(Z_DIM),
        )
    def forward(self, x):
        h = self.emb(x).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            z = torch.nn.functional.gelu(conv(h))
            z = torch.max(z, dim=-1).values
            pooled.append(z)
        return self.proj(torch.cat(pooled, dim=1))

class MLPEncoder(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(256, Z_DIM),
            nn.GELU(),
            nn.LayerNorm(Z_DIM),
        )
    def forward(self, x):
        return self.net(x)

class FusionModel(nn.Module):
    def __init__(self, vocab_size, dims):
        super().__init__()
        self.token = TokenEncoder(vocab_size)
        self.branches = nn.ModuleDict()
        for name, d in dims.items():
            self.branches[name] = MLPEncoder(d)
        n = 128 * (1 + len(dims))
        self.fusion = nn.Sequential(
            nn.Linear(n, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(256, Z_DIM),
            nn.GELU(),
            nn.LayerNorm(Z_DIM),
        )
        self.head = nn.Linear(Z_DIM, 1)

    def forward(self, tok, branch_inputs):
        zs = [self.token(tok)]
        for name in self.branches:
            zs.append(self.branches[name](branch_inputs[name]))
        z = self.fusion(torch.cat(zs, dim=1))
        return self.head(z).squeeze(1), z

def train_model(model, train_loader, val_loader, pos_weight):
    model = model.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight], device=DEVICE)
    )
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_auc = -np.inf
    best_state = None
    best_epoch = 0
    bad = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in train_loader:
            tok = batch[0].to(DEVICE)
            y = batch[1].to(DEVICE)
            branch_inputs = {
                k: v.to(DEVICE) for k, v in batch[2].items()
            } if len(batch) > 2 else {}
            opt.zero_grad(set_to_none=True)
            logits, _ = model(tok, branch_inputs)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        probs, ys = [], []
        with torch.no_grad():
            for batch in val_loader:
                tok = batch[0].to(DEVICE)
                y = batch[1].cpu().numpy()
                branch_inputs = {
                    k: v.to(DEVICE) for k, v in batch[2].items()
                }
                logits, _ = model(tok, branch_inputs)
                probs.extend(torch.sigmoid(logits).cpu().numpy())
                ys.extend(y)

        auc = roc_auc_score(ys, probs)
        print(f"epoch {epoch:02d} | val AUC {auc:.5f}")
        if auc > best_auc:
            best_auc = auc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return model, best_epoch, best_auc

def get_embeddings(model, loader):
    model.eval()
    Z, P, Y = [], [], []
    with torch.no_grad():
        for batch in loader:
            tok = batch[0].to(DEVICE)
            y = batch[1].numpy()
            branch_inputs = {k: v.to(DEVICE) for k, v in batch[2].items()}
            logits, z = model(tok, branch_inputs)
            Z.append(z.cpu().numpy())
            P.append(torch.sigmoid(logits).cpu().numpy())
            Y.append(y)
    return np.vstack(Z), np.concatenate(P), np.concatenate(Y)

def best_threshold(y, p):
    ts = np.arange(0.10, 0.6001, 0.005)
    accs = [accuracy_score(y, p >= t) for t in ts]
    j = int(np.argmax(accs))
    return float(ts[j]), float(accs[j])

def probe_metrics(Ztr, ytr, Zv, yv):
    svm = LinearSVC(C=1.0)
    svm.fit(Ztr, ytr)
    svm_score = svm.decision_function(Zv)
    svm_acc = accuracy_score(yv, svm_score >= 0)
    svm_auc = roc_auc_score(yv, svm_score)

    lr = LogisticRegression(max_iter=2000, C=1.0)
    lr.fit(Ztr, ytr)
    lr_score = lr.predict_proba(Zv)[:,1]
    lr_acc = accuracy_score(yv, lr_score >= 0.5)
    lr_auc = roc_auc_score(yv, lr_score)

    return svm_acc, svm_auc, lr_acc, lr_auc

def between_within_ratio(Z, y):
    z0 = Z[y == 0]
    z1 = Z[y == 1]
    c0, c1 = z0.mean(0), z1.mean(0)
    between = np.linalg.norm(c0 - c1)
    within = 0.5 * (
        np.mean(np.linalg.norm(z0 - c0, axis=1)) +
        np.mean(np.linalg.norm(z1 - c1, axis=1))
    )
    return float(between), float(within), float(between / max(within, 1e-12))


## Run experiments

Run the experiment cells one at a time. Each cell uses the same canonical split and saves its own outputs. The SVM/LR probes are fitted on **training embeddings only** and evaluated on held-out validation embeddings.

In [9]:

# ==============================
# Exp10: TF-IDF + Transition TF-IDF
# ==============================
start_time = time.time()

# Token branch: vocabulary is built ONLY from the training fold.
train_docs = df.iloc[train_idx]["tokens"].tolist()
val_docs = df.iloc[val_idx]["tokens"].tolist()
y_train = df.iloc[train_idx]["label_num"].values.astype(np.float32)
y_val = df.iloc[val_idx]["label_num"].values.astype(np.float32)

vocab = build_vocab(train_docs)
tok_train = encode_docs(train_docs, vocab)
tok_val = encode_docs(val_docs, vocab)


# Token TF-IDF branch
token_strings_train = [" ".join(d) for d in train_docs]
token_strings_val = [" ".join(d) for d in val_docs]

vec_tok = TfidfVectorizer(
    ngram_range=(1,6),
    min_df=TFIDF_MIN_DF,
    max_features=TFIDF_MAX_FEATURES,
    sublinear_tf=True,
    token_pattern=r"(?u)\S+",
    dtype=np.float32,
)
A = vec_tok.fit_transform(token_strings_train)
B = vec_tok.transform(token_strings_val)
svd_tok = TruncatedSVD(n_components=min(SVD_DIM, A.shape[1]-1), random_state=SEED)
A = svd_tok.fit_transform(A).astype(np.float32)
B = svd_tok.transform(B).astype(np.float32)

# Transition TF-IDF branch
tr_train = make_transition_strings(train_docs)
tr_val = make_transition_strings(val_docs)
vec_tr = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=TRANS_MIN_DF,
    max_features=TRANS_MAX_FEATURES,
    sublinear_tf=True,
    token_pattern=r"(?u)\S+",
    dtype=np.float32,
)
C = vec_tr.fit_transform(tr_train)
D = vec_tr.transform(tr_val)
svd_tr = TruncatedSVD(n_components=min(SVD_DIM, C.shape[1]-1), random_state=SEED)
C = svd_tr.fit_transform(C).astype(np.float32)
D = svd_tr.transform(D).astype(np.float32)

branch_train = {"tfidf": A, "transition_tfidf": C}
branch_val = {"tfidf": B, "transition_tfidf": D}
print("Token TF-IDF:", A.shape, B.shape)
print("Transition TF-IDF:", C.shape, D.shape)


class MultiInputDataset(Dataset):
    def __init__(self, tok, y, branch_arrays):
        self.tok = tok
        self.y = np.asarray(y, dtype=np.float32)
        self.b = branch_arrays
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return (torch.tensor(self.tok[i], dtype=torch.long),
                torch.tensor(self.y[i], dtype=torch.float32),
                {k: torch.tensor(v[i], dtype=torch.float32) for k,v in self.b.items()})

train_ds = MultiInputDataset(tok_train, y_train, branch_train)
val_ds   = MultiInputDataset(tok_val, y_val, branch_val)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

dims = {k: v.shape[1] for k,v in branch_train.items()}
model = FusionModel(len(vocab), dims)
pos_weight = float((y_train == 0).sum() / max(1, (y_train == 1).sum()))

model, best_epoch, val_auc = train_model(model, train_loader, val_loader, pos_weight)

Ztr, ptr, ytr = get_embeddings(model, train_loader)
Zv, pv, yv = get_embeddings(model, val_loader)

thr, best_acc = best_threshold(yv, pv)
acc05 = accuracy_score(yv, pv >= 0.5)
svm_acc, svm_auc, lr_acc, lr_auc = probe_metrics(Ztr, ytr, Zv, yv)
between, within, ratio = between_within_ratio(Zv, yv)

result = {
    "experiment": "Exp10",
    "title": "TF-IDF + Transition TF-IDF",
    "val_accuracy_0.5": float(acc05),
    "val_accuracy_best_threshold": float(best_acc),
    "best_threshold": float(thr),
    "val_auc_neural": float(val_auc),
    "svm_accuracy": float(svm_acc),
    "svm_auc": float(svm_auc),
    "lr_accuracy": float(lr_acc),
    "lr_auc": float(lr_auc),
    "between": float(between),
    "within": float(within),
    "between_within_ratio": float(ratio),
    "best_epoch": int(best_epoch),
    "params": int(sum(p.numel() for p in model.parameters())),
    "time_min": float((time.time() - start_time)/60),
}

print("\nRESULT")
print(pd.Series(result))

safe = "Exp10".lower()
with open(os.path.join(OUTPUT_DIR, f"{safe}_validation_result.json"), "w") as f:
    json.dump(result, f, indent=2)
np.savez_compressed(
    os.path.join(OUTPUT_DIR, f"{safe}_validation_embeddings.npz"),
    Z_train=Ztr, Z_val=Zv, y_train=ytr, y_val=yv,
    p_val=pv,
)
torch.save(
    model.state_dict(),
    os.path.join(OUTPUT_DIR, f"{safe}_best_validation_model.pt")
)



Token TF-IDF: (8428, 256) (2108, 256)
Transition TF-IDF: (8428, 256) (2108, 256)
epoch 01 | val AUC 0.91125
epoch 02 | val AUC 0.93666
epoch 03 | val AUC 0.93619
epoch 04 | val AUC 0.92509
epoch 05 | val AUC 0.93927
epoch 06 | val AUC 0.94106
epoch 07 | val AUC 0.94309
epoch 08 | val AUC 0.93304
epoch 09 | val AUC 0.93836
epoch 10 | val AUC 0.92979

RESULT
experiment                                          Exp10
title                          TF-IDF + Transition TF-IDF
val_accuracy_0.5                                 0.879032
val_accuracy_best_threshold                      0.887097
best_threshold                                      0.145
val_auc_neural                                   0.943095
svm_accuracy                                     0.861006
svm_auc                                          0.927777
lr_accuracy                                      0.860057
lr_auc                                           0.942352
between                                         12.935838
wit

In [11]:

# ==============================
# Exp11: TF-IDF + Structural
# ==============================
start_time = time.time()

# Token branch: vocabulary is built ONLY from the training fold.
train_docs = df.iloc[train_idx]["tokens"].tolist()
val_docs = df.iloc[val_idx]["tokens"].tolist()
y_train = df.iloc[train_idx]["label_num"].values.astype(np.float32)
y_val = df.iloc[val_idx]["label_num"].values.astype(np.float32)

vocab = build_vocab(train_docs)
tok_train = encode_docs(train_docs, vocab)
tok_val = encode_docs(val_docs, vocab)


token_strings_train = [" ".join(d) for d in train_docs]
token_strings_val = [" ".join(d) for d in val_docs]

vec_tok = TfidfVectorizer(
    ngram_range=(1,6),
    min_df=TFIDF_MIN_DF,
    max_features=TFIDF_MAX_FEATURES,
    sublinear_tf=True,
    token_pattern=r"(?u)\S+",
    dtype=np.float32,
)
A = vec_tok.fit_transform(token_strings_train)
B = vec_tok.transform(token_strings_val)
svd_tok = TruncatedSVD(n_components=min(SVD_DIM, A.shape[1]-1), random_state=SEED)
A = svd_tok.fit_transform(A).astype(np.float32)
B = svd_tok.transform(B).astype(np.float32)

# Helper wrapper to avoid the assertion 62 error in case the base function hasn't been re-executed yet
def safe_sequence_features(d):
    try:
        return sequence_features(d)
    except AssertionError:
        # If it asserted on 62 but returned 58, extract the features without the assertion
        # by temporarily catching and returning the generated features.
        # To be fully robust, let's just compute features and truncate/ignore the assertion:
        import inspect
        # We can dynamically run the body of sequence_features or adjust the assertion count.
        # Since the assertion failed at the end of the function, we can retrieve the list via traceback
        # or redefine a safe version here.
        pass

# Let's define a safe local version of sequence_features without the strict length check
def local_sequence_features(doc):
    x = list(doc)
    n = len(x)
    if n == 0:
        return np.zeros(58, dtype=np.float32)

    cnt = Counter(x)
    uniq = len(cnt)

    bigrams = list(zip(x[:-1], x[1:]))
    trigrams = list(zip(x[:-2], x[1:-1], x[2:]))

    def div(seq):
        return len(set(seq)) / max(1, len(seq))

    def rep(seq):
        return 1.0 - div(seq)

    half = max(1, n // 2)
    q = max(1, n // 4)

    first = x[:half]
    second = x[half:]
    q1, q2, q3, q4 = x[:q], x[q:2*q], x[2*q:3*q], x[3*q:]

    feats = [
        n,
        uniq,
        uniq / n,
        len([v for v in cnt.values() if v > 1]) / n,
        max(cnt.values()) / n,
        np.mean(list(cnt.values())),
        np.std(list(cnt.values())),
        entropy_from_counts(list(cnt.values())),
        len(bigrams),
        len(set(bigrams)),
        div(bigrams),
        rep(bigrams),
        len(trigrams),
        len(set(trigrams)),
        div(trigrams),
        rep(trigrams),
        sum(x[i] == x[i-1] for i in range(1, n)) / max(1, n-1),
        len(set(zip(x[:-1], x[1:]))) / max(1, n-1),
        entropy_from_counts(list(Counter(bigrams).values())),
        entropy_from_counts(list(Counter(trigrams).values())),
        len(set(first)) / len(first),
        len(set(second)) / max(1, len(second)),
        entropy_from_counts(list(Counter(first).values())),
        entropy_from_counts(list(Counter(second).values())),
        len(set(q1)) / len(q1),
        len(set(q2)) / len(q2),
        len(set(q3)) / len(q3),
        len(set(q4)) / max(1, len(q4)),
        entropy_from_counts(list(Counter(q1).values())),
        entropy_from_counts(list(Counter(q2).values())),
        entropy_from_counts(list(Counter(q3).values())),
        entropy_from_counts(list(Counter(q4).values())),
        len(first),
        len(second),
        len(q1),
        len(q2),
        len(q3),
        len(q4),
        len(set(first) & set(second)) / max(1, len(set(first) | set(second))),
        len(set(q1) & set(q2)) / max(1, len(set(q1) | set(q2))),
        len(set(q2) & set(q3)) / max(1, len(set(q2) | set(q3))),
        len(set(q3) & set(q4)) / max(1, len(set(q3) | set(q4))),
        np.mean([len(t) for t in x]),
        np.std([len(t) for t in x]),
        min([len(t) for t in x]),
        max([len(t) for t in x]),
        np.mean([len(t) for t in x[:half]]),
        np.mean([len(t) for t in x[half:]]) if second else 0.0,
        len(x[0]),
        len(x[-1]),
        len(set(x[:min(10,n)])) / min(10,n),
        len(set(x[max(0,n-10):])) / min(10,n),
        entropy_from_counts(list(Counter(x[:min(10,n)]).values())),
        entropy_from_counts(list(Counter(x[max(0,n-10):]).values())),
        len(set(x[:min(25,n)])) / min(25,n),
        len(set(x[max(0,n-25):])) / min(25,n),
        entropy_from_counts(list(Counter(x[:min(25,n)]).values())),
        entropy_from_counts(list(Counter(x[max(0,n-25):]).values())),
    ]
    return np.asarray(feats, dtype=np.float32)

X = np.vstack([local_sequence_features(d) for d in train_docs + val_docs])
scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

branch_train = {
    "tfidf": A,
    "structural": X[:len(train_docs)],
}
branch_val = {
    "tfidf": B,
    "structural": X[len(train_docs):],
}
print("Token TF-IDF:", A.shape, B.shape)
print("Structural:", branch_train["structural"].shape, branch_val["structural"].shape)


class MultiInputDataset(Dataset):
    def __init__(self, tok, y, branch_arrays):
        self.tok = tok
        self.y = np.asarray(y, dtype=np.float32)
        self.b = branch_arrays
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return (torch.tensor(self.tok[i], dtype=torch.long),
                torch.tensor(self.y[i], dtype=torch.float32),
                {k: torch.tensor(v[i], dtype=torch.float32) for k,v in self.b.items()})

train_ds = MultiInputDataset(tok_train, y_train, branch_train)
val_ds   = MultiInputDataset(tok_val, y_val, branch_val)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

dims = {k: v.shape[1] for k,v in branch_train.items()}
model = FusionModel(len(vocab), dims)
pos_weight = float((y_train == 0).sum() / max(1, (y_train == 1).sum()))

model, best_epoch, val_auc = train_model(model, train_loader, val_loader, pos_weight)

Ztr, ptr, ytr = get_embeddings(model, train_loader)
Zv, pv, yv = get_embeddings(model, val_loader)

thr, best_acc = best_threshold(yv, pv)
acc05 = accuracy_score(yv, pv >= 0.5)
svm_acc, svm_auc, lr_acc, lr_auc = probe_metrics(Ztr, ytr, Zv, yv)
between, within, ratio = between_within_ratio(Zv, yv)

result = {
    "experiment": "Exp11",
    "title": "TF-IDF + Structural",
    "val_accuracy_0.5": float(acc05),
    "val_accuracy_best_threshold": float(best_acc),
    "best_threshold": float(thr),
    "val_auc_neural": float(val_auc),
    "svm_accuracy": float(svm_acc),
    "svm_auc": float(svm_auc),
    "lr_accuracy": float(lr_acc),
    "lr_auc": float(lr_auc),
    "between": float(between),
    "within": float(within),
    "between_within_ratio": float(ratio),
    "best_epoch": int(best_epoch),
    "params": int(sum(p.numel() for p in model.parameters())),
    "time_min": float((time.time() - start_time)/60),
}

print("\nRESULT")
print(pd.Series(result))

safe = "Exp11".lower()
with open(os.path.join(OUTPUT_DIR, f"{safe}_validation_result.json"), "w") as f:
    json.dump(result, f, indent=2)
np.savez_compressed(
    os.path.join(OUTPUT_DIR, f"{safe}_validation_embeddings.npz"),
    Z_train=Ztr, Z_val=Zv, y_train=ytr, y_val=yv,
    p_val=pv,
)
torch.save(
    model.state_dict(),
    os.path.join(OUTPUT_DIR, f"{safe}_best_validation_model.pt")
)


Token TF-IDF: (8428, 256) (2108, 256)
Structural: (8428, 58) (2108, 58)
epoch 01 | val AUC 0.94350
epoch 02 | val AUC 0.95188
epoch 03 | val AUC 0.95673
epoch 04 | val AUC 0.96126
epoch 05 | val AUC 0.95693
epoch 06 | val AUC 0.95427
epoch 07 | val AUC 0.95588

RESULT
experiment                                   Exp11
title                          TF-IDF + Structural
val_accuracy_0.5                          0.887097
val_accuracy_best_threshold               0.894213
best_threshold                               0.555
val_auc_neural                            0.961259
svm_accuracy                              0.890892
svm_auc                                   0.950105
lr_accuracy                               0.889943
lr_auc                                    0.958637
between                                  11.750198
within                                    4.249189
between_within_ratio                       2.76528
best_epoch                                       4
params           

In [13]:

# ==============================
# Exp12: Transition TF-IDF + Structural
# ==============================
start_time = time.time()

# Token branch: vocabulary is built ONLY from the training fold.
train_docs = df.iloc[train_idx]["tokens"].tolist()
val_docs = df.iloc[val_idx]["tokens"].tolist()
y_train = df.iloc[train_idx]["label_num"].values.astype(np.float32)
y_val = df.iloc[val_idx]["label_num"].values.astype(np.float32)

vocab = build_vocab(train_docs)
tok_train = encode_docs(train_docs, vocab)
tok_val = encode_docs(val_docs, vocab)


tr_train = make_transition_strings(train_docs)
tr_val = make_transition_strings(val_docs)
vec_tr = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=TRANS_MIN_DF,
    max_features=TRANS_MAX_FEATURES,
    sublinear_tf=True,
    token_pattern=r"(?u)\S+",
    dtype=np.float32,
)
A = vec_tr.fit_transform(tr_train)
B = vec_tr.transform(tr_val)
svd_tr = TruncatedSVD(n_components=min(SVD_DIM, A.shape[1]-1), random_state=SEED)
A = svd_tr.fit_transform(A).astype(np.float32)
B = svd_tr.transform(B).astype(np.float32)

# Define a safe local version of sequence_features without the strict length check
def local_sequence_features(doc):
    x = list(doc)
    n = len(x)
    if n == 0:
        return np.zeros(58, dtype=np.float32)

    cnt = Counter(x)
    uniq = len(cnt)

    bigrams = list(zip(x[:-1], x[1:]))
    trigrams = list(zip(x[:-2], x[1:-1], x[2:]))

    def div(seq):
        return len(set(seq)) / max(1, len(seq))

    def rep(seq):
        return 1.0 - div(seq)

    half = max(1, n // 2)
    q = max(1, n // 4)

    first = x[:half]
    second = x[half:]
    q1, q2, q3, q4 = x[:q], x[q:2*q], x[2*q:3*q], x[3*q:]

    feats = [
        n,
        uniq,
        uniq / n,
        len([v for v in cnt.values() if v > 1]) / n,
        max(cnt.values()) / n,
        np.mean(list(cnt.values())),
        np.std(list(cnt.values())),
        entropy_from_counts(list(cnt.values())),
        len(bigrams),
        len(set(bigrams)),
        div(bigrams),
        rep(bigrams),
        len(trigrams),
        len(set(trigrams)),
        div(trigrams),
        rep(trigrams),
        sum(x[i] == x[i-1] for i in range(1, n)) / max(1, n-1),
        len(set(zip(x[:-1], x[1:]))) / max(1, n-1),
        entropy_from_counts(list(Counter(bigrams).values())),
        entropy_from_counts(list(Counter(trigrams).values())),
        len(set(first)) / len(first),
        len(set(second)) / max(1, len(second)),
        entropy_from_counts(list(Counter(first).values())),
        entropy_from_counts(list(Counter(second).values())),
        len(set(q1)) / len(q1),
        len(set(q2)) / len(q2),
        len(set(q3)) / len(q3),
        len(set(q4)) / max(1, len(q4)),
        entropy_from_counts(list(Counter(q1).values())),
        entropy_from_counts(list(Counter(q2).values())),
        entropy_from_counts(list(Counter(q3).values())),
        entropy_from_counts(list(Counter(q4).values())),
        len(first),
        len(second),
        len(q1),
        len(q2),
        len(q3),
        len(q4),
        len(set(first) & set(second)) / max(1, len(set(first) | set(second))),
        len(set(q1) & set(q2)) / max(1, len(set(q1) | set(q2))),
        len(set(q2) & set(q3)) / max(1, len(set(q2) | set(q3))),
        len(set(q3) & set(q4)) / max(1, len(set(q3) | set(q4))),
        np.mean([len(t) for t in x]),
        np.std([len(t) for t in x]),
        min([len(t) for t in x]),
        max([len(t) for t in x]),
        np.mean([len(t) for t in x[:half]]),
        np.mean([len(t) for t in x[half:]]) if second else 0.0,
        len(x[0]),
        len(x[-1]),
        len(set(x[:min(10,n)])) / min(10,n),
        len(set(x[max(0,n-10):])) / min(10,n),
        entropy_from_counts(list(Counter(x[:min(10,n)]).values())),
        entropy_from_counts(list(Counter(x[max(0,n-10):]).values())),
        len(set(x[:min(25,n)])) / min(25,n),
        len(set(x[max(0,n-25):])) / min(25,n),
        entropy_from_counts(list(Counter(x[:min(25,n)]).values())),
        entropy_from_counts(list(Counter(x[max(0,n-25):]).values())),
    ]
    return np.asarray(feats, dtype=np.float32)

X = np.vstack([local_sequence_features(d) for d in train_docs + val_docs])
scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

branch_train = {
    "transition_tfidf": A,
    "structural": X[:len(train_docs)],
}
branch_val = {
    "transition_tfidf": B,
    "structural": X[len(train_docs):],
}
print("Transition TF-IDF:", A.shape, B.shape)
print("Structural:", branch_train["structural"].shape, branch_val["structural"].shape)


class MultiInputDataset(Dataset):
    def __init__(self, tok, y, branch_arrays):
        self.tok = tok
        self.y = np.asarray(y, dtype=np.float32)
        self.b = branch_arrays
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return (torch.tensor(self.tok[i], dtype=torch.long),
                torch.tensor(self.y[i], dtype=torch.float32),
                {k: torch.tensor(v[i], dtype=torch.float32) for k,v in self.b.items()})

train_ds = MultiInputDataset(tok_train, y_train, branch_train)
val_ds   = MultiInputDataset(tok_val, y_val, branch_val)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

dims = {k: v.shape[1] for k,v in branch_train.items()}
model = FusionModel(len(vocab), dims)
pos_weight = float((y_train == 0).sum() / max(1, (y_train == 1).sum()))

model, best_epoch, val_auc = train_model(model, train_loader, val_loader, pos_weight)

Ztr, ptr, ytr = get_embeddings(model, train_loader)
Zv, pv, yv = get_embeddings(model, val_loader)

thr, best_acc = best_threshold(yv, pv)
acc05 = accuracy_score(yv, pv >= 0.5)
svm_acc, svm_auc, lr_acc, lr_auc = probe_metrics(Ztr, ytr, Zv, yv)
between, within, ratio = between_within_ratio(Zv, yv)

result = {
    "experiment": "Exp12",
    "title": "Transition TF-IDF + Structural",
    "val_accuracy_0.5": float(acc05),
    "val_accuracy_best_threshold": float(best_acc),
    "best_threshold": float(thr),
    "val_auc_neural": float(val_auc),
    "svm_accuracy": float(svm_acc),
    "svm_auc": float(svm_auc),
    "lr_accuracy": float(lr_acc),
    "lr_auc": float(lr_auc),
    "between": float(between),
    "within": float(within),
    "between_within_ratio": float(ratio),
    "best_epoch": int(best_epoch),
    "params": int(sum(p.numel() for p in model.parameters())),
    "time_min": float((time.time() - start_time)/60),
}

print("\nRESULT")
print(pd.Series(result))

safe = "Exp12".lower()
with open(os.path.join(OUTPUT_DIR, f"{safe}_validation_result.json"), "w") as f:
    json.dump(result, f, indent=2)
np.savez_compressed(
    os.path.join(OUTPUT_DIR, f"{safe}_validation_embeddings.npz"),
    Z_train=Ztr, Z_val=Zv, y_train=ytr, y_val=yv,
    p_val=pv,
)
torch.save(
    model.state_dict(),
    os.path.join(OUTPUT_DIR, f"{safe}_best_validation_model.pt")
)


Transition TF-IDF: (8428, 256) (2108, 256)
Structural: (8428, 58) (2108, 58)
epoch 01 | val AUC 0.93384
epoch 02 | val AUC 0.93600
epoch 03 | val AUC 0.95353
epoch 04 | val AUC 0.94894
epoch 05 | val AUC 0.95277
epoch 06 | val AUC 0.94343

RESULT
experiment                                              Exp12
title                          Transition TF-IDF + Structural
val_accuracy_0.5                                     0.883302
val_accuracy_best_threshold                          0.890417
best_threshold                                           0.58
val_auc_neural                                       0.953531
svm_accuracy                                         0.889469
svm_auc                                              0.943931
lr_accuracy                                          0.887097
lr_auc                                               0.950775
between                                             10.901049
within                                               4.472155
between_w

In [14]:

# ==============================
# Combined summary
# ==============================
results = []
for fn in sorted(os.listdir(OUTPUT_DIR)):
    if fn.endswith("_validation_result.json"):
        with open(os.path.join(OUTPUT_DIR, fn), "r") as f:
            results.append(json.load(f))

summary = pd.DataFrame(results)
if len(summary):
    summary = summary.sort_values("val_accuracy_best_threshold", ascending=False)
    display(summary)
    summary.to_csv(os.path.join(OUTPUT_DIR, "phase2_validation_summary.csv"), index=False)
else:
    print("No result JSONs found yet. Run the experiment cells first.")

print("\nOutputs are in:", OUTPUT_DIR)


,experiment,title,val_accuracy_0.5,val_accuracy_best_threshold,best_threshold,val_auc_neural,svm_accuracy,svm_auc,lr_accuracy,lr_auc,between,within,between_within_ratio,best_epoch,params,time_min
1,Exp11,TF-IDF + Structural,0.887097,0.894213,0.555,0.961259,0.890892,0.950105,0.889943,0.958637,11.750198,4.249189,2.765280,4,2347905,1.369670
2,Exp12,Transition TF-IDF + Structural,0.883302,0.890417,0.580,0.953531,0.889469,0.943931,0.887097,0.950775,10.901049,4.472155,2.437538,3,2347905,0.736533
0,Exp10,TF-IDF + Transition TF-IDF,0.879032,0.887097,0.145,0.943095,0.861006,0.927777,0.860057,0.942352,12.935838,5.406851,2.392490,7,2398593,1.717770



Outputs are in: /content/phase2_outputs


In [16]:
import shutil
from google.colab import files

# Compress the directory into a zip file
shutil.make_archive('/content/phase2_outputs', 'zip', '/content/phase2_outputs_10,11,12')

# Download the file to your local computer
files.download('/content/phase2_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## After the comparison

Do **not** use the validation result as Kaggle test accuracy. After choosing a promising configuration, a separate final-training notebook/script should fit the chosen pipeline on all labelled training examples and only then predict the 3,000 unlabeled test examples.